<a href="https://colab.research.google.com/github/bzhuang2-create/SURF---MC-Simulation-for-Educational-Research/blob/main/Energy%2C_Pressure%2C_Initialization_and_Iteration_Helpers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Single and Total Potential Energy Calculations
'''
Calculates all interaction energies for a single atom, excluding itself

atom_numbers: list of atomic numbers
atom_positions: list of atomic positions
no_atoms: total number of atoms
box_length: length of the cube (A)
selected: the index of the atom to calculate energies for
energy_fxn: a function to calculate pairwise interaction energies. Must be of the form fxn(radius, atom#1, atom#2, params)
params: any additional parameters the user-specified energy function needs

returns: the total interaction energies for a single atom
'''
@njit(fastmath = True)
def single_energy(atom_numbers, atom_positions, no_atoms, box_length, selected, energy_fxn, params):
  # Calculates the interaction energies for a selected atom

  energy = 0
  for i in range(no_atoms):
    if(i != selected):
      fixed_position = atom_positions[selected]
      other_position = atom_positions[i]

      dx = fixed_position[0] - other_position[0]
      dy = fixed_position[1] - other_position[1]
      dz = fixed_position[2] - other_position[2]

      # enforce periodic boundary conditions
      dx -= box_length * round(dx / box_length)
      dy -= box_length * round(dy / box_length)
      dz -= box_length * round(dz / box_length)
      r = np.sqrt(dx * dx + dy * dy + dz * dz)

      energy += energy_fxn(r, atom_numbers[selected], atom_numbers[i], params)

  return energy

'''
Calculates the total energy of the system by finding all pairwise interaction energies and dividing by 2 to avoid double-counting

atom_numbers: list of atomic numbers
atom_positions: list of atomic positions
no_atoms: total number of atoms
box_length: length of the cube (A)
energy_fxn: a function to calculate pairwise interaction energies. Must be of the form fxn(radius, atom#1, atom#2, params)
params: any additional parameters the user-specified energy function needs

returns: the total interaction energies for a system
'''
@njit(fastmath = True)
def total_energy(atom_numbers, atom_positions, no_atoms, box_length, energy_fxn, params):
  energy = 0
  for i in range(no_atoms):
    energy += single_energy(atom_numbers, atom_positions, no_atoms, box_length, i, energy_fxn, params)

  return energy / 2

In [ ]:
# @title Pressure Calculations and Partial RDFs

'''
Converts a distance to a bin index for an RDF

distance: the distance to convert
R_bins: a list of distances corresponding to the RDF

returns: the index of the bin the distance is in
'''
@njit(fastmath = True)
def distance_to_bin(distance, R_bins):
  single_radius = R_bins[1] - R_bins[0]
  # this assumes there are at least two bins, which there should be if the slider minimums are appropriate

  index = np.floor(distance / single_radius)
  return int(index)


'''
Generates a partial RDF for a specific pair of species in the system. Does not check both directions

selected_species: the atomic symbol for the first atom
target_species: the atomic symbol for the second atom
species_list: list of names of atoms (ex. ['H', 'H', 'C'])
no_atoms: total number of atoms
atom_positions: list of atomic positions, corresponding to species_list
box_length: length of the cube (A)
R_bins: a list of distances corresponding to the RDF

returns: a single partial RDF for the two species
'''
@njit(fastmath = True)
def partial_RDF(selected_species, target_species, species_list, no_atoms, atom_positions, box_length, R_bins):
  histogram = np.zeros(len(R_bins))
  a_select_count = 0
  b_select_count = 0
  volume = box_length ** 3

  for i in range(no_atoms):
    if(species_list[i] == selected_species):
      selected_position = atom_positions[i]
      a_select_count += 1
      b_select_count = 0

      for j in range(no_atoms):
        if(species_list[j] == target_species):
          target_position = atom_positions[j]
          b_select_count += 1

          dx = selected_position[0] - target_position[0]
          dy = selected_position[1] - target_position[1]
          dz = selected_position[2] - target_position[2]

          dx -= box_length * round(dx / box_length)
          dy -= box_length * round(dy / box_length)
          dz -= box_length * round(dz / box_length)

          distance = np.sqrt(dx*dx + dy*dy + dz*dz)
          index = distance_to_bin(distance, R_bins)

          if(index < len(histogram)):
            histogram[index] += 1

  histogram = histogram / max(a_select_count * b_select_count, 1) * volume
  for i in range(len(histogram)):
    radius = R_bins[i]
    if(radius == 0):
      histogram[i] = 0
    else:
      histogram[i] = histogram[i] / (4 * np.pi * (radius ** 2) * (R_bins[1] - R_bins[0]))

  return histogram


'''
Generates a list of bucket distances for an RDF given the length of the box and the number of bins

box_length: length of the cube (A)
no_bins: number of bins

returns: a list of distances corresponding for an RDF
'''
@njit(fastmath = True)
def bins_to_distance(box_length, no_bins):
  distances = np.zeros(no_bins)
  for i in range(no_bins):
    distances[i] = i * (box_length / 2.01) / no_bins
  return distances


'''
This is the radial distribution function that is assumed at very low densities
g(r) = exp(-E(r) / kB * T)

radius: the distance between the two atoms
temperature: the temperature of the system (K)
atom1: the atomic number of the first atom
atom2: the atomic number of the second atom
params: any additional parameters the user-specified energy function needs
energy_fxn: a function to calculate pairwise interaction energies. Must be of the form fxn(radius, atom#1, atom#2, params)

returns: the value of the radial distribution function
'''
@njit(fastmath = True)
def mayer_f_fxn(radius, temperature, atom1, atom2, params, energy_fxn):
  if (radius <= 0):
    radius = 1e-6
  return np.exp(-energy_fxn(radius, atom1, atom2, params) / (kB * temperature))


'''
Calculates the various integrands of the pressure equation. Does NOT include weights by molar fraction or the 2/3(pi) coefficient in front

atom1: atomic number of the first atom type
atom2: atomic number of the second atom type
RDF: a list corresponding to the RDF for the specific atom1/atom2 combination
R_bins: a list of distances corresponding to the RDF
params: any additional parameters the user-specified energy function needs
energy_fxn: a function to calculate pairwise interaction energies. Must be of the form fxn(radius, atom#1, atom#2, params)
energy_fxn_derivative: the derivative of energy_fxn w/r/t r. Must be of the form fxn(radius, atom#1, atom#2, params)
mayer: whether to use the mayer f-fxn simplification for calculating pressure

returns: the value of the integrand
'''
@njit(fastmath = True)
def pressure_equation(temperature, atom1, atom2, RDF, R_bins, params, energy_fxn = lennard_jones, energy_fxn_derivative = lennard_jones_derivative, mayer = False):
  sum = 0
  for i in range(len(R_bins) - 1):
    r1 = R_bins[i]
    r2 = R_bins[i + 1]
    if(r1 == 0 or r2 == 0):
      continue

    dEdr2 = energy_fxn_derivative(r2, atom1, atom2, params)
    dEdr1 = energy_fxn_derivative(r1, atom1, atom2, params)

    if(mayer):
      gr2 = mayer_f_fxn(r2, temperature, atom1, atom2, params, energy_fxn)
      gr1 = mayer_f_fxn(r1, temperature, atom1, atom2, params, energy_fxn)
    else:
      gr2 = RDF[i + 1]
      gr1 = RDF[i]

    total2 = (r2 ** 3) * gr2 * dEdr2
    total1 = (r1 ** 3) * gr1 * dEdr1

    sum += (total1 + total2) * (r2 - r1) / 2

  return sum

'''
Given a species string and a fraction_list tuple, find the mol fraction of species

fraction_list: list of tuples of the type (Atomic Symbol, Mol Fraction)
species: string of the species to find the fraction of

returns: the mol fraction of the species
'''
@njit(fastmath = True)
def find_fractions(fraction_list, species):
  for i in range(len(fraction_list)):
    if(fraction_list[i][0] == species):
      return fraction_list[i][1]

'''
Note: by default, the cutoff radius is box_length / 2.01. If this is changed anywhere, need to update it in the partial RDF generation as well

tail_correction_fxn: a function to calculate the tail correction factor. Must be of the form fxn(density1, density2, atom#1, atom#2, params, cutoff_radius)
atom1: the atomic number of the first atom
atom2: the atomic number of the second atom
fraction1: the mol fraction of the first atom
fraction2: the mol fraction of the second atom
params: any additional parameters the user-specified energy function needs
cutoff_radius: the cutoff radius (A)
density: the density of the system (atoms per A^3)

returns: the tail correction factor
'''
def tail_correction(tail_correction_fxn, atom1, atom2, fraction1, fraction2, params, cutoff_radius, density):
  return tail_correction_fxn(fraction1 * density, fraction2 * density, atom1, atom2, params, cutoff_radius)


'''
Given a distance and an energy function, calculates the pairwise force between two atoms

r: the distance between the two atoms (A)
atom1: the atomic number of the first atom
atom2: the atomic number of the second atom
params: any additional parameters the user-specified energy function needs

returns: the pairwise force between the two atoms (eV/A)
'''
@njit(fastmath = True)
def pairwise_force(r, atom1, atom2, params):
  dEdr = lennard_jones_derivative(r, atom1, atom2, params)
  return dEdr


'''
calculates the 1/(3V)... term in the virial pressure equation

atom_numbers: list of atomic numbers
atom_positions: list of atomic positions
box_length: length of the cube (A)
energy_fxn: a function to calculate pairwise interaction energies. Must be of the form fxn(radius, atom#1, atom#2, params)
params: any additional parameters the user-specified energy function needs

returns: the 1/(3V) term in the virial pressure equation (eV)
'''
@njit(fastmath = True)
def virial_pressure(atom_numbers, atom_positions, box_length, energy_fxn, params):
  sum = 0

  for i in range(len(atom_numbers)):
    for j in range(len(atom_numbers)):
      if(i != j):

        atom1 = atom_numbers[i]
        atom2 = atom_numbers[j]

        dx = atom_positions[i][0] - atom_positions[j][0]
        dy = atom_positions[i][1] - atom_positions[j][1]
        dz = atom_positions[i][2] - atom_positions[j][2]

        dx -= box_length * round(dx / box_length)
        dy -= box_length * round(dy / box_length)
        dz -= box_length * round(dz / box_length)

        r = np.sqrt(dx * dx + dy * dy + dz * dz)
        force = pairwise_force(r, atom1, atom2, params)
        sum += r * force

  return sum / (6 * box_length ** 3)
  #dividing by 6 avoids double counting

In [ ]:
# @title Simulation Helpers and Initialization

'''
Converts a given number of atoms and number density to the corresponding cube length

no_atoms: total number of atoms
rho: density (atoms per A^3)
returns: the length of the cube (A)
'''
@njit(fastmath = True)
def density_to_length(no_atoms, rho):
  Volume = no_atoms / rho
  Length = np.cbrt(Volume)
  return Length

'''
Initializes an ASE Atoms object for the Monte Carlo Simulation

no_atoms: total number of atoms
length: length of the cube
species: list of tuples of the type (Atomic Symbol, Mol Fraction)
reference_start: whether to use the equilibrium reference positions

returns: an ASE Atoms object, with all atoms placed in random locations inside the box
'''
def initialize_atoms(no_atoms, length, species, reference_start):

  position_count = 0
  string_count = 0

  if(reference_start):
    positions = []
    reference_length = density_to_length(reference_no_atoms, reference_density)
    scaling_factor = length / reference_length

    for i in range(min(no_atoms, reference_no_atoms)):
      positions.append([reference_positions[i][0] * scaling_factor, reference_positions[i][1] * scaling_factor, reference_positions[i][2] * scaling_factor])
      position_count += 1

    for i in range(no_atoms - reference_no_atoms):
      positions.append([np.random.rand() * length, np.random.rand() * length, np.random.rand() * length])
      position_count += 1

  else:
    positions = ((np.random.rand(no_atoms, 3)) * length).tolist()
    position_count += no_atoms

  species_string = ""
  for i in range(len(species)):
    add_count = int(species[i][1] * no_atoms)
    species_string += species[i][0] * add_count
    string_count += add_count

  # final wrap-ups to account for potential rounding errors
  # will always use the last species in the species array (not a significant imbalance for most systems)
  while(position_count < string_count):
    positions.append([np.random.rand() * length, np.random.rand() * length, np.random.rand() * length])
    #species_string += species[-1][0]
    position_count += 1

  while(string_count < position_count):
    species_string += species[-1][0]
    string_count += 1

  while(string_count < no_atoms):
    species_string += species[0][0]
    positions.append([np.random.rand() * length, np.random.rand() * length, np.random.rand() * length])
    string_count += 1

  assert(position_count == len(positions), f"position_count: {position_count} len:{len(positions)}")
  MC = Atoms(species_string, positions, cell = [length, length, length], pbc = [True, True, True])
  return MC


'''
Initializes an ASE Atoms object for the Monte Carlo Simulation with all atoms starting on a cubic lattice structure (or as close as possible)

no_atoms: total number of atoms
length: length of the cube
species: list of tuples of the type (Atomic Symbol, Mol Fraction)

returns: an ASE Atoms object, with all atoms placed in a cubic lattice
'''
def initialize_atoms_cube(no_atoms, length, species):
  dimen = int(np.ceil(no_atoms ** (1/3)))
  step = length / dimen
  positions = np.zeros((no_atoms, 3))

  species_string = ""
  for i in range(len(species)):
    species_string += species[i][0] * int(species[i][1] * no_atoms)

  for i in range(dimen):
    for j in range(dimen):
      for k in range(dimen):
        index = i * dimen ** 2 + j * dimen + k
        if(index < no_atoms):
          positions[index] = [i * step, j * step, k * step]

  MC = Atoms(species_string, positions, cell = [length, length, length], pbc = [True, True, True])
  return MC

'''
Initializes the monte Carlo simulation

no_atoms: total number of atoms
density: number density of the atoms (atoms per A^3)
species: list of tuples of the type (Atomic Symbol, Mol Fraction)
reference_start: whether to use the equilibrium reference positions

returns: an ASE Atoms object, with all atoms placed in random locations inside the box
'''
def MC_initial(no_atoms, density, species, cube_start, reference_start):
  length = density_to_length(no_atoms, density)

  # this means that cueb_start will override reference_start
  if(cube_start):
    atoms = initialize_atoms_cube(no_atoms, length, species)
  else:
    atoms = initialize_atoms(no_atoms, length, species, reference_start)

  return atoms

'''
Creates a unit vector pointing in a random direction
returns: a numpy array serving as the unit vector
'''
@njit(fastmath=True)
def unit():
  vector = np.array([np.random.uniform(-1, 1), np.random.uniform(-1, 1), np.random.uniform(-1, 1)])
  return vector / np.linalg.norm(vector)


'''
Iterates the Monte Carlo simulation according to the Metropolis algorithm

no_atoms: total number of atoms
atom_numbers: list of atomic numbers
current_positions: list of atomic positions
length: length of the cube
energy_fxn: a function to calculate pairwise interaction energies. Must be of the form fxn(radius, atom#1, atom#2, params)
params: any additional parameters the user-specified energy function needs
step_size: the size of the step to take
n_warmup: the number of iterations to take before energy values are recorded for heat capacity
energy_array: an array to append total system energy values to
temperature: temperature (K)
i: the current iteration number
total_potential_energy: the total potential energy of the system
move_accepted_array: an array to append whether or not a move was accepted
append: whether or not to append to the energy array

returns: the updated positions, total potential energy, energy array (iucluding the updated total potential energy), and move accepted array
'''
def iterate(no_atoms, atom_numbers, current_positions, length, energy_fxn, params, step_size, n_warmup, energy_array, temperature, i, total_potential_energy, move_accepted_array, append):
  selected_atom = random.randrange(0, no_atoms, 1)
  current_energy = single_energy(atom_numbers, current_positions, no_atoms, length, selected_atom, energy_fxn, params)

  vector = np.random.normal(0, step_size, 3)
  current_positions[selected_atom] += vector
  current_positions[selected_atom] %= length

  new_energy = single_energy(atom_numbers, current_positions, no_atoms, length, selected_atom, energy_fxn, params)
  energy_difference = new_energy - current_energy
  new_potential_energy = total_potential_energy

  if(energy_difference < 0):
    new_potential_energy += energy_difference
    move_accepted_array[i] = True
  else:
    probability = np.exp(-((energy_difference) / (temperature * kB)))
    if(np.random.rand() < probability):
      new_potential_energy += energy_difference
      move_accepted_array[i] = True
    else:
      current_positions[selected_atom] -= vector
      current_positions[selected_atom] %= length
      move_accepted_array[i] = False

  if(i > n_warmup and append):
    energy_array.append(new_potential_energy)

  return current_positions, new_potential_energy, energy_array, move_accepted_array

In [ ]:
# @title Energy-Related Helpers

'''
Converts a potential energy value into a bin for the histogram

energy: the potential energy to convert (eV)
bin_size: the size of the bins to use for the histogram (eV)
cutoff_bin: this bin corresponds to zero energy (the bin before this index is the lowest negative energy bin, and this bin in the lowest positive energy bin)

returns: the index of the bin to place the energy in
'''
@njit(fastmath = True)
def energy_to_bin(energy, bin_size, cutoff_bin):

  index_raw = int(energy / bin_size)
  index_adjust = index_raw + cutoff_bin
  return index_adjust

'''
Finds the energy distribution of a system, accounting for all atoms

bin_size: the size of the bins to use for the histogram
no_bins: the number of bins to use for the histogram
cutoff_bins: this bin corresponds to zero energy (the bin before this index is the lowest negative energy bin, and this bin in the lowest positive energy bin)
no_atoms: total number of atoms
atom_positions: list of atom positions
atom_numbers: list of atomic numbers
box_length: length of the cube (A)
energy_fxn: a function to calculate pairwise interaction energies. Must be of the form fxn(radius, atom#1, atom#2, params)
params: any additional parameters the user-specified energy function needs

returns: the normalized histogram, bin size, number of bins, and cutoff bin
'''
def find_energy_distribution(bin_size, no_bins, cutoff_bin, no_atoms, atom_positions, atom_numbers, box_length, energy_fxn, params):
  histogram = np.zeros(no_bins)

  for i in range(no_atoms):
    energy = single_energy(atom_numbers, atom_positions, no_atoms, box_length, i, energy_fxn, params)

    index = energy_to_bin(energy, bin_size, cutoff_bin)
    if(index < no_bins):
      histogram[index] += 1
    elif(index >= no_bins):
      histogram[-1] += 1
    else:
      histogram[0] += 1

  return (histogram / no_atoms, bin_size, no_bins, cutoff_bin)

'''
Finds the energy distribution of a system, accounting for all atoms without normalization or binning to a histogram

atom_positions: list of atom positions
atom_numbers: list of atomic numbers
box_length: length of the cube (A)
energy_fxn: a function to calculate pairwise interaction energies. Must be of the form fxn(radius, atom#1, atom#2, params)
params: any additional parameters the user-specified energy function needs

returns: an array with the potential energy values for all atoms
'''
def find_energy_distribution_raw(no_atoms, atom_positions, atom_numbers, box_length, energy_fxn, params):
  energy_array = []

  for i in range(no_atoms):
    energy = single_energy(atom_numbers, atom_positions, no_atoms, box_length, i, energy_fxn, params)
    energy_array.append(energy)

  return energy_array